# 3. Príznaky klipov pomocou video siete R(2+1)D

ResNet18 spracúva každú snímku samostatne a nevidí pohyb. Jedenie a pitie sa však líšia
práve pohybom (ruka k ústam, nakláňanie pohára, žuvanie).

**R(2+1)D-18** je konvolučná sieť pre video. Každú 3D konvolúciu rozkladá na
2D konvolúciu v priestore a 1D konvolúciu v čase, takže spracúva celý klip naraz.
Je predtrénovaná na datasete **Kinetics-400** (400 ľudských činností), medzi ktorými sú
aj rôzne druhy jedenia a pitia.

- Poslednú vrstvu `fc` nahradíme `nn.Identity()` → **jeden vektor 512 príznakov na celý klip**.
- Sieť je zmrazená.
- Vstup: 16 snímok, veľkosť 112 × 112, normalizácia Kinetics. Všetko robí `weights.transforms()`.

Výsledok: `features_r21d.npy` s tvarom `(počet klipov, 512)`, použitý v notebooku 6.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

from har_utils import read_frames, to_video_tensor

df = pd.read_csv("index.csv")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
weights = R2Plus1D_18_Weights.KINETICS400_V1
model = r2plus1d_18(weights=weights)
model.fc = nn.Identity()            # namiesto 400 tried Kinetics -> 512 príznakov
model.eval().to(device)
video_transform = weights.transforms()
print(video_transform)

In [ ]:
feats = []
with torch.no_grad():
    for path in df.path:
        x = to_video_tensor(read_frames(path), video_transform).to(device)   # (1, 3, 16, 112, 112)
        feats.append(model(x).squeeze(0).cpu().numpy())                      # (512,)

feats = np.stack(feats)
np.save("features_r21d.npy", feats)
print(feats.shape)